# Step 2 - Train YOLOv8 on Your Own Labeled Dataset

In [7]:
!pip install labelImg

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 8.5 MB/s eta 0:00:01
   --------------- ------------------------ 1.6/4.0 MB 4.2 MB/s eta 0:00:01
   -------------------- ------------------- 2.1/4.0 MB 3.8 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.0 MB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 4.1 MB/s  0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ------- -------------------------------- 1.3/6.9 MB 8.4 MB/s eta 0:00:01
   ------------------ --------------------- 3.1/6.9 MB 8.0 MB/s eta 0:00:01
   

In [4]:
import shutil
import os
import random
import yaml
from pathlib import Path

In [49]:
from ultralytics import YOLO
import os

IMAGES_DIR = r"D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\images"
LABELS_DIR = r"D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\labels"

os.makedirs(LABELS_DIR, exist_ok=True)

model = YOLO("yolov8n.pt")

results = model.predict(
    source=IMAGES_DIR,
    save=False,
    conf=0.25,
    verbose=False
)

for r in results:
    img_path = r.path
    img_name = os.path.splitext(os.path.basename(img_path))[0]

    label_file = os.path.join(LABELS_DIR, img_name + ".txt")

    with open(label_file, "w") as f:
        if r.boxes is not None:
            for box in r.boxes:
                cls = int(box.cls[0])
                x_center, y_center, w, h = box.xywhn[0].tolist()

                if cls == 0:
                    f.write(f"0 {x_center} {y_center} {w} {h}\n")

In [50]:
IMAGES_DIR = r'D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\images'

LABELS_DIR = r'D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\labels'

DATASET_DIR = r'D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset'

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

CLASS_NAMES = ['player']

EPOCHS      = 50  
IMAGE_SIZE  = 416  
BATCH_SIZE  = 4   
BASE_MODEL  = 'yolov8n.pt' 

RANDOM_SEED = 42

random.seed(RANDOM_SEED)

In [51]:
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DATASET_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR, 'labels', split), exist_ok=True)

In [54]:
image_files = sorted([
    f for f in os.listdir(IMAGES_DIR)
    if f.endswith('.jpg') or f.endswith('.png')
])

paired = []
skipped = []

for img_file in image_files:
    label_file = img_file.rsplit('.', 1)[0] + '.txt'
    label_path = os.path.join(LABELS_DIR, label_file)

    if os.path.exists(label_path):
        paired.append((img_file, label_file))
    else:
        skipped.append(img_file)

print(f'Images with labels - {len(paired)}')
print(f'Images without labels (skipped) - {len(skipped)}')

Images with labels - 312
Images without labels (skipped) - 0


In [55]:
random.shuffle(paired)

n       = len(paired)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train_set = paired[:n_train]
val_set   = paired[n_train : n_train + n_val]
test_set  = paired[n_train + n_val:]

print(f'Split summary')
print(f'  Train - {len(train_set)} images')
print(f'  Val   - {len(val_set)} images')
print(f'  Test  - {len(test_set)} images')
print(f'  Total - {n} images')

Split summary
  Train - 218 images
  Val   - 46 images
  Test  - 48 images
  Total - 312 images


In [56]:
def copy_split(pairs, split_name):
    img_out = os.path.join(DATASET_DIR, 'images', split_name)
    lbl_out = os.path.join(DATASET_DIR, 'labels', split_name)

    for img_file, lbl_file in pairs:
        shutil.copy(
            os.path.join(IMAGES_DIR, img_file),
            os.path.join(img_out,    img_file)
        )
        shutil.copy(
            os.path.join(LABELS_DIR, lbl_file),
            os.path.join(lbl_out,    lbl_file)
        )

copy_split(train_set, 'train')
copy_split(val_set,   'val')
copy_split(test_set,  'test')

In [ ]:
yaml_content = {
    'path'   : DATASET_DIR.replace('\\', '/'),
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : len(CLASS_NAMES),
    'names' : CLASS_NAMES,
}

yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f'data.yaml written to: {yaml_path}')
print()
print('Contents:')
with open(yaml_path) as f:
    print(f.read())

data.yaml written to: D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\data.yaml

Contents:
names:
- player
nc: 1
path: D:/MSc/DS 5216 - Artificial Intelligence/PAS02_DTS2430/dataset
test: images/test
train: images/train
val: images/val



In [59]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

print(f'  Base model - {BASE_MODEL}')
print(f'  Epochs     - {EPOCHS}')
print(f'  Image size - {IMAGE_SIZE}')
print(f'  Batch size - {BATCH_SIZE}')
print(f'  Device     - CPU')


results = model.train(
    data      = yaml_path,
    epochs    = EPOCHS,
    imgsz     = IMAGE_SIZE,
    batch     = BATCH_SIZE,
    device    = 'cpu',
    project   = os.path.join(DATASET_DIR, 'runs'),
    name      = 'sports_player_detection',
    exist_ok  = True,
    patience  = 15,       
    save      = True,
    plots     = True,      
    verbose   = True,
)


  Base model - yolov8n.pt
  Epochs     - 50
  Image size - 416
  Batch size - 4
  Device     - CPU
Ultralytics 8.4.78  Python-3.10.0 torch-2.12.1+cpu CPU (13th Gen Intel Core i7-13620H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train

In [60]:
from ultralytics import YOLO

model = YOLO(
    r"D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\runs\sports_player_detection\weights\best.pt"
)

results = model.predict(
    source=r"D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\Football\Football 01.mp4",
    conf=0.25,
    save=True
)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/311) D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\Football\Football 01.mp4: 256x416 16 players, 33.9ms
video 1/1 (frame 2/311) D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\Football\Football 01.mp4: 256x416 16 players, 16.1ms
video 1/1 (frame 3/311) D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\Football\Football 01.mp4: 256x416 17 players, 22.3ms
video 1/1 (frame 4/311) D:\MSc\DS 5216 - Artificial Intelligence\

In [61]:
run_dir = os.path.join(DATASET_DIR, 'runs', 'sports_player_detection')

print('Training outputs saved to:')
print(f'  {run_dir}')
print()
print('Key files generated automatically by Ultralytics:')
print('  results.png          — loss curves, precision, recall, mAP over epochs')
print('  PR_curve.png         — Precision-Recall curve')
print('  confusion_matrix.png — confusion matrix')
print('  weights/best.pt      — best model weights (use this for inference)')
print('  weights/last.pt      — last epoch weights')
print()

if os.path.exists(run_dir):
    files = os.listdir(run_dir)
    print('Files found:')
    for f in sorted(files):
        print(f'  {f}')
else:
    print('Run directory not found yet.')

Training outputs saved to:
  D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\runs\sports_player_detection

Key files generated automatically by Ultralytics:
  results.png          — loss curves, precision, recall, mAP over epochs
  PR_curve.png         — Precision-Recall curve
  confusion_matrix.png — confusion matrix
  weights/best.pt      — best model weights (use this for inference)
  weights/last.pt      — last epoch weights

Files found:
  BoxF1_curve.png
  BoxPR_curve.png
  BoxP_curve.png
  BoxR_curve.png
  args.yaml
  confusion_matrix.png
  confusion_matrix_normalized.png
  labels.jpg
  results.csv
  results.png
  train_batch0.jpg
  train_batch1.jpg
  train_batch2.jpg
  train_batch2200.jpg
  train_batch2201.jpg
  train_batch2202.jpg
  val_batch0_labels.jpg
  val_batch0_pred.jpg
  val_batch1_labels.jpg
  val_batch1_pred.jpg
  val_batch2_labels.jpg
  val_batch2_pred.jpg
  weights


In [63]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plots_to_show = [
    ('results.png',          'Training Metrics (Loss, Precision, Recall, mAP)'),
    ('BoxPR_curve.png',         'Precision-Recall Curve'),
    ('confusion_matrix.png', 'Confusion Matrix'),
]

for fname, title in plots_to_show:
    fpath = os.path.join(run_dir, fname)
    if os.path.exists(fpath):
        img = mpimg.imread(fpath)
        plt.figure(figsize=(12, 6))
        plt.imshow(img)
        plt.title(title, fontsize=13, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Not found: {fpath}')

<Figure size 1200x600 with 1 Axes>

<Figure size 1200x600 with 1 Axes>

<Figure size 1200x600 with 1 Axes>

In [64]:
best_weights = os.path.join(run_dir, 'weights', 'best.pt')

if not os.path.exists(best_weights):
    print('best.pt not found. Training may not have completed.')
else:
    trained_model = YOLO(best_weights)

    metrics = trained_model.val(
        data   = yaml_path,
        split  = 'test',
        device = 'cpu',
        verbose= True,
    )

    print('\nTEST SET RESULTS')
    print(f'  mAP@0.50      : {metrics.box.map50:.4f}')
    print(f'  mAP@0.50:0.95 : {metrics.box.map:.4f}')
    print(f'  Precision     : {metrics.box.mp:.4f}')
    print(f'  Recall        : {metrics.box.mr:.4f}')

Ultralytics 8.4.78  Python-3.10.0 torch-2.12.1+cpu CPU (13th Gen Intel Core i7-13620H)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 689.5446.3 MB/s, size: 366.0 KB)
val: Scanning D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\labels\test... 48 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 48/48 1.5Kit/s 0.0s
val: New cache created: D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.1it/s 1.4s0.9s
                   all         48        614      0.931       0.86      0.884      0.651
Speed: 0.2ms preprocess, 13.9ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\Code_PAS02\runs\detect\val

TEST SET RESULTS
  mAP@0.50      : 0.8842
  mAP@0.50:0.95 : 0.6510
  Precision 

In [65]:
import pandas as pd

if os.path.exists(best_weights):
    test_metrics = pd.DataFrame([{
        'Model'         : 'YOLOv8n (fine-tuned)',
        'Dataset'       : 'Custom sports dataset',
        'Classes'       : 'player',
        'Train images'  : len(train_set),
        'Val images'    : len(val_set),
        'Test images'   : len(test_set),
        'Epochs'        : EPOCHS,
        'Image size'    : IMAGE_SIZE,
        'Precision'     : round(metrics.box.mp,  4),
        'Recall'        : round(metrics.box.mr,  4),
        'mAP@0.50'      : round(metrics.box.map50, 4),
        'mAP@0.50:0.95' : round(metrics.box.map,   4),
    }])

    csv_out = os.path.join(DATASET_DIR, 'test_results.csv')
    test_metrics.to_csv(csv_out, index=False)

    print(test_metrics.to_string(index=False))
    print(f'\nSaved to: {csv_out}')

               Model               Dataset Classes  Train images  Val images  Test images  Epochs  Image size  Precision  Recall  mAP@0.50  mAP@0.50:0.95
YOLOv8n (fine-tuned) Custom sports dataset  player           218          46           48      50         416     0.9309  0.8599    0.8842          0.651

Saved to: D:\MSc\DS 5216 - Artificial Intelligence\PAS02_DTS2430\dataset\test_results.csv
